# Final Walk-Forward Retention Contract

Executable evidence that final SOXL actor retention requires strict walk-forward pass, candidate-filter actor retention, and fresh SOXLBacktest buy/sell baseline alignment.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'SPEC.md').exists():
    ROOT = ROOT.parent
if not (ROOT / 'SPEC.md').exists():
    raise RuntimeError('Could not locate SOXL workspace root from notebook cwd')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.final_walkforward_retention_contract import (
    derive_final_walkforward_retention_contract,
    load_latest_reference_baseline,
    render_final_walkforward_retention_contract,
    write_final_walkforward_retention_contract,
)

CARD = ROOT / 'runtime/model_selection/card/final_walkforward_retention_contract_v1'
CARD.mkdir(parents=True, exist_ok=True)

STRICT_GATE_PATH = ROOT / 'runtime/model_selection/card/model_retention_walkforward_gate_v1/summary.json'
CANDIDATE_FILTER_PATH = ROOT / 'runtime/model_selection/card/strict_candidate_retention_filter_v1/summary.json'
REFERENCE_METRICS_PATH = ROOT / 'runtime/production/card/soxl_backtest_policy_reimplementation_v1/legacy_comparison_metrics.csv'

SOURCES = [
    ('strict_walkforward_gate', STRICT_GATE_PATH, 'strict walk-forward drawdown and return gate rows'),
    ('strict_candidate_retention_filter', CANDIDATE_FILTER_PATH, 'actor-retention allow/deny filter'),
    ('latest_soxlbacktest_reference', REFERENCE_METRICS_PATH, 'fresh SOXLBacktest buy/sell baseline metrics'),
]
manifest = pd.DataFrame([
    {
        'source_label': label,
        'path': str(path.relative_to(ROOT)),
        'exists': path.exists(),
        'role': role,
    }
    for label, path, role in SOURCES
])
manifest

,source_label,path,exists,role
0,strict_walkforward_gate,runtime/model_selection/card/model_retention_w...,True,strict walk-forward drawdown and return gate rows
1,strict_candidate_retention_filter,runtime/model_selection/card/strict_candidate_...,True,actor-retention allow/deny filter
2,latest_soxlbacktest_reference,runtime/production/card/soxl_backtest_policy_r...,True,fresh SOXLBacktest buy/sell baseline metrics


In [2]:
missing = manifest.loc[~manifest['exists'], 'path'].tolist()
if missing:
    raise FileNotFoundError(f'Missing required sources: {missing}')

strict_gate_payload = json.loads(STRICT_GATE_PATH.read_text(encoding='utf-8'))
candidate_filter_payload = json.loads(CANDIDATE_FILTER_PATH.read_text(encoding='utf-8'))
latest_reference_baseline = load_latest_reference_baseline(ROOT)

result = derive_final_walkforward_retention_contract(
    strict_gate_payload=strict_gate_payload,
    strict_candidate_filter_payload=candidate_filter_payload,
    latest_reference_baseline=latest_reference_baseline,
)
rows = pd.DataFrame(result['rows'])
summary = result['summary']
summary

{'decision': 'final_walkforward_retention_contract_no_retained_models',
 'reference_baseline': 'soxl_backtest_policy_reimplementation_v1',
 'reference_return_pct': 146.043798,
 'latest_reference_return_pct': 146.04379795135492,
 'reference_is_current': True,
 'global_blockers': [],
 'strict_passing_profile_count': 0,
 'candidate_filter_actor_retention_allowed_count': 0,
 'final_retained_count': 0,
 'final_retained_profiles': [],
 'current_best_return_pct': 224.361539,
 'required_best_return_pct': 213.143462,
 'max_walkforward_drawdown_pct': 20.0,
 'required_wf_windows': ['wf_2020', 'wf_2022', 'wf_2023_2024']}

In [3]:
display_columns = [
    'profile',
    'strict_gate_pass',
    'promotion_allowed',
    'actor_retention_allowed',
    'final_model_retained',
    'blockers',
    'ytd_return_pct',
    'required_best_return_pct',
    'required_soxlbacktest_return_pct',
    'worst_wf_max_drawdown_pct',
    'missing_wf_window_count',
]
rows[display_columns].sort_values(['final_model_retained', 'strict_gate_pass', 'profile'], ascending=[False, False, True])

,profile,strict_gate_pass,promotion_allowed,actor_retention_allowed,final_model_retained,blockers,ytd_return_pct,required_best_return_pct,required_soxlbacktest_return_pct,worst_wf_max_drawdown_pct,missing_wf_window_count
1,balanced,False,False,False,False,[missing_wf_window:wf_2020; missing_wf_window:...,159.205721,213.143462,146.043798,NaN,3
2,balanced_macro_v1,False,False,False,False,[missing_wf_window:wf_2020; missing_wf_window:...,196.749008,213.143462,146.043798,NaN,3
6,balanced_rule_residual_imitation_v4,False,False,False,False,[wf_max_drawdown_ge_20:wf_2020; wf_max_drawdow...,145.031235,213.143462,146.043798,35.710864,0
3,balanced_rule_residual_v19,False,False,False,False,[wf_max_drawdown_ge_20:wf_2020; wf_max_drawdow...,173.299227,213.143462,146.043798,46.653957,0
10,balanced_rule_residual_v19_reg,False,False,False,False,[wf_max_drawdown_ge_20:wf_2022; wf_max_drawdow...,141.992184,213.143462,146.043798,46.171637,0
11,balanced_rule_residual_v19_reg_light,False,False,False,False,[wf_max_drawdown_ge_20:wf_2020; wf_max_drawdow...,164.253638,213.143462,146.043798,51.596776,0
12,balanced_rule_residual_v19_reg_v22_retest_v1,False,False,False,False,[missing_wf_window:wf_2020; missing_wf_window:...,175.012377,213.143462,146.043798,NaN,3
5,balanced_rule_residual_v22,False,False,False,False,[wf_max_drawdown_ge_20:wf_2020; wf_max_drawdow...,116.882644,213.143462,146.043798,54.665610,0
7,drawdown_rebound_reward_train_v1,False,False,False,False,[wf_max_drawdown_ge_20:wf_2020; ytd_return_bel...,172.832548,213.143462,146.043798,23.546830,0
8,mode_conditioned_rebound_reward_train_v2,False,False,False,False,[wf_max_drawdown_ge_20:wf_2020; wf_max_drawdow...,177.573569,213.143462,146.043798,21.831720,0


In [4]:
sidecars = write_final_walkforward_retention_contract(result, output_dir=CARD)
manifest_path = CARD / 'source_manifest.csv'
manifest.to_csv(manifest_path, index=False)
result['source_manifest'] = manifest.to_dict(orient='records')
result['sidecars'] = {
    **{key: str(path.relative_to(ROOT)) for key, path in sidecars.items()},
    'source_manifest_csv': str(manifest_path.relative_to(ROOT)),
}
(CARD / 'summary.json').write_text(json.dumps(result, indent=2, sort_keys=True) + '\n', encoding='utf-8')
print(render_final_walkforward_retention_contract(result))

# Final Walk-Forward Retention Contract

- decision: `final_walkforward_retention_contract_no_retained_models`
- final_retained_count: `0`
- reference_baseline: `soxl_backtest_policy_reimplementation_v1`
- reference_is_current: `True`
- strict_passing_profile_count: `0`

## Rows

- `soxl_backtest_policy_reimplementation_v1`: final_retained=False, actor_retention=False, blockers=`SOXLBacktest reference baseline cannot be retained as actor`
- `balanced`: final_retained=False, actor_retention=False, blockers=`missing_wf_window:wf_2020; missing_wf_window:wf_2022; missing_wf_window:wf_2023_2024; ytd_return_below_best_alignment`
- `balanced_macro_v1`: final_retained=False, actor_retention=False, blockers=`missing_wf_window:wf_2020; missing_wf_window:wf_2022; missing_wf_window:wf_2023_2024; ytd_return_below_best_alignment`
- `balanced_rule_residual_v19`: final_retained=False, actor_retention=False, blockers=`wf_max_drawdown_ge_20:wf_2020; wf_max_drawdown_ge_20:wf_2022; wf_max_drawdown_ge_20:w

## Conclusion

- Run: `final_walkforward_retention_contract_v1` combined strict WF retention gate, strict candidate retention filter, and latest SOXLBacktest buy/sell reference.
- Decision: `final_walkforward_retention_contract_no_retained_models`; final_retained_count=`0`; final_retained_profiles=`[]`.
- Key gates: strict_passing_profile_count=`0`, candidate_filter_actor_retention_allowed_count=`0`, max_walkforward_drawdown_pct=`20.0`.
- Baseline freshness: reference_return_pct=`146.043798`, latest_reference_return_pct=`146.04379795135492`, reference_is_current=`True`.
- Risk note: SOXLBacktest remains a reference baseline only, not a runnable actor; current model pool has no final-retainable actor under the WF <20% drawdown and best-return alignment contract.
- Artifacts: `summary.json`, `final_retention_contract.csv`, `source_manifest.csv`, and `final_walkforward_retention_contract.md` in this card.
- Next action: train or select a model that passes all three WF windows, worst WF MaxDD <20%, YTD return >= current-best alignment threshold, and SOXLBacktest reference return.